# AI-Powered Study Assistant

The AI-Powered Study Assistant is an intelligent system that helps students interact with their study materials efficiently. Using Retrieval-Augmented Generation, it allows users to upload documents and ask questions in natural language. The system processes documents by extracting, chunking, and converting text into embeddings stored in a vector database. When a query is asked, it retrieves relevant content and generates accurate, context-based answers using a language model. This reduces manual searching, saves time, and improves learning. The system is scalable, user-friendly, and can be extended with features like summarization, voice interaction, and personalized recommendations.

## 1. Install and Import Dependencies

In [9]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [17]:
%pip install -U sentence-transformers huggingface-hub transformers

  Using cached sentence_transformers-5.6.0-py3-none-any.whl.metadata (18 kB)
Using cached sentence_transformers-5.6.0-py3-none-any.whl (596 kB)
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/11.6 MB 12.0 MB/s eta 0:00:01
   -------------- ------------------------- 4.2/11.6 MB 12.6 MB/s eta 0:00:01
   ------------------ --------------------- 5.5/11.6 MB 10.2 MB/s eta 0:00:01
   ---------------------- ----------------- 6.6/11.6 MB 8.4 MB/s eta 0:00:01
   ------------------------- -------------- 7.3/11.6 MB 8.0 MB/s eta 0:00:01
   --------------------------- ------------ 8.1/11.6 MB 7.0 MB/s eta 0:00:01
   ---------------------------- ----------- 8.4/11.6 MB 6.5 MB/s eta 0:00:01
   ------------------------------ --------- 8.9/11.6 MB 5.7 MB/s eta 0:00:01
   ------------------------------- -------- 9.2/11.6 MB 5.2 MB/s eta 0:00:01
   -------------------------------- ------- 9.4/11.6 MB 4.9 MB/s eta 0:00:01
   ----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires pydantic<=3.0,>=2.0, but you have pydantic 1.10.26 which is incompatible.


In [10]:
from pathlib import Path
import re

from langchain.document_loaders import PyPDFLoader, TextLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS

In [11]:
BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
STORE_DIR = BASE_DIR / "store"

DATA_DIR.mkdir(exist_ok=True)
STORE_DIR.mkdir(exist_ok=True)

print("Data folder:", DATA_DIR)
print("Store folder:", STORE_DIR)

Data folder: D:\Desktop Folders\GitHub Content\Gaurang_Gupta_JECRC_Foundation_CEI\project_AI_Study_Assistant\data
Store folder: D:\Desktop Folders\GitHub Content\Gaurang_Gupta_JECRC_Foundation_CEI\project_AI_Study_Assistant\store


In [12]:
def load_documents(folder_path: Path):
    documents = []

    if not folder_path.exists():
        return documents

    for file_path in folder_path.rglob("*"):
        if not file_path.is_file():
            continue

        suffix = file_path.suffix.lower()

        try:
            if suffix == ".pdf":
                loader = PyPDFLoader(str(file_path))
                documents.extend(loader.load())
            elif suffix == ".txt":
                loader = TextLoader(str(file_path), encoding="utf-8")
                documents.extend(loader.load())
        except Exception as exc:
            print(f"Skipping {file_path.name}: {exc}")

    return documents


docs = load_documents(DATA_DIR)
print(f"Loaded {len(docs)} document pages from raw files")

Loaded 1 document pages from raw files


In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
)

chunks = text_splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

Created 7 chunks


In [16]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence_transformers`.

In [ ]:
if len(chunks) > 0:
    vectorstore = FAISS.from_documents(chunks, embedding_model)
    vectorstore.save_local(str(STORE_DIR))
    print("Vector store created and saved successfully")
else:
    print("No document chunks available. Add PDF or TXT files to the data folder first.")

In [ ]:
if (STORE_DIR / "index.faiss").exists() and (STORE_DIR / "index.pkl").exists():
    vectorstore = FAISS.load_local(
        str(STORE_DIR),
        embedding_model,
        allow_dangerous_deserialization=True,
    )
    print("Vector store loaded successfully")
else:
    print("Vector store not found. Run the previous cell first.")

In [ ]:
def normalize_text(text: str) -> str:
    return " ".join(text.split()).strip()


def build_short_snippet(text: str, limit: int = 1000) -> str:
    snippet = normalize_text(text)
    if len(snippet) <= limit:
        return snippet
    return snippet[:limit].rsplit(" ", 1)[0] + "..."


def answer_question(question, k=4):
    if "vectorstore" not in globals():
        return "Vector store is not loaded."

    relevant_docs = vectorstore.similarity_search(question, k=k)

    if not relevant_docs:
        return "I could not find relevant information in the uploaded documents."

    question_terms = {
        term.lower()
        for term in re.findall(r"[A-Za-z0-9]+", question)
        if len(term) > 2
    }

    scored_sentences = []

    for doc in relevant_docs:
        sentences = re.split(r"(?<=[.!?])\s+", doc.page_content)
        for sentence in sentences:
            clean_sentence = normalize_text(sentence)
            if not clean_sentence:
                continue

            score = sum(1 for term in question_terms if term in clean_sentence.lower())
            if score > 0:
                scored_sentences.append((score, clean_sentence))

    scored_sentences.sort(key=lambda item: item[0], reverse=True)

    selected_sentences = []
    seen_sentences = set()

    for _, sentence in scored_sentences:
        if sentence in seen_sentences:
            continue
        selected_sentences.append(sentence)
        seen_sentences.add(sentence)
        if len(selected_sentences) == 3:
            break

    if selected_sentences:
        return "\n\n".join(f"- {sentence}" for sentence in selected_sentences)

    return build_short_snippet(relevant_docs[0].page_content)

In [ ]:
sample_question = "What is Retrieval-Augmented Generation?"
print(answer_question(sample_question))

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    question_input = widgets.Text(
        value="What is Retrieval-Augmented Generation?",
        description="Question:",
        layout=widgets.Layout(width="80%"),
    )

    output = widgets.Output()

    def on_change(change):
        with output:
            output.clear_output()
            print("Question:", question_input.value)
            print()
            print(answer_question(question_input.value))

    question_input.observe(on_change, names="value")
    display(question_input, output)

except Exception as exc:
    print("ipywidgets not available:", exc)

In [ ]:
print("AI-Powered Study Assistant notebook completed.")
print("Flow:")
print("1. Load documents")
print("2. Split into chunks")
print("3. Create embeddings")
print("4. Save to FAISS vector store")
print("5. Retrieve relevant context for user questions")